# Late Delivery Risk Prediction — Full MLOps Pipeline

**Business context:** 54.83% of orders in the DataCo supply chain dataset are delivered late.
This notebook demonstrates the complete ML pipeline: data loading with leakage protection,
model comparison, SHAP explainability, and MLflow experiment tracking.

**Models compared:**
1. Logistic Regression (interpretable baseline)
2. XGBoost — default (max_depth=6, n_estimators=100)
3. XGBoost — shallow (max_depth=3, n_estimators=200)
4. XGBoost — deep (max_depth=8, n_estimators=100)

**Primary metric:** F1 score (accuracy is misleading with 55/45 class split)

In [ ]:
import sys
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from adapters.data.csv_repository import DataCoCSVRepository

# Use sample for quick demo, swap to full path for production metrics
DATA_PATH = PROJECT_ROOT / "data" / "sample" / "sample.csv"
# DATA_PATH = PROJECT_ROOT / "data" / "raw" / "DataCoSupplyChainDataset.csv"

repo = DataCoCSVRepository(DATA_PATH)
orders = repo.get_orders()

late = sum(1 for o in orders if o.late_delivery_risk == 1)
print(f"Loaded {len(orders)} orders")
print(f"Late delivery rate: {late/len(orders):.1%}")
print("Leakage columns stripped by adapter (Days for shipping real, Delivery Status, shipping date)")

## Step 1: Train Baseline — Logistic Regression

Logistic Regression provides an interpretable baseline. Odds ratios from coefficients
tell us which features increase/decrease risk. This is the model a business stakeholder
can reason about.

In [ ]:
from adapters.ml.feature_encoder import FeatureEncoder
from adapters.ml.mlflow_tracker import MLflowTracker
from adapters.ml.shap_explainer import ShapExplainer
from adapters.ml.sklearn_predictor import (
    LogisticRegressionPredictor,
    XGBoostPredictor,
)
from application.use_cases import TrainAndEvaluateUseCase

TRACKING_URI = f"sqlite:///{PROJECT_ROOT}/mlflow.db"

configs = [
    ("logreg-baseline", LogisticRegressionPredictor()),
    ("xgboost-default", XGBoostPredictor(n_estimators=100, max_depth=6, learning_rate=0.1)),
    ("xgboost-shallow", XGBoostPredictor(n_estimators=200, max_depth=3, learning_rate=0.1)),
    ("xgboost-deep", XGBoostPredictor(n_estimators=100, max_depth=8, learning_rate=0.05)),
]

results = {}
for run_name, model in configs:
    print(f"\n--- Training: {run_name} ---")
    tracker = MLflowTracker(experiment_name="late-delivery-risk", tracking_uri=TRACKING_URI)
    use_case = TrainAndEvaluateUseCase(
        data_repo=DataCoCSVRepository(DATA_PATH),
        feature_encoder=FeatureEncoder(),
        model=model,
        explainer_factory=lambda m, names: ShapExplainer(m.model, names),
        tracker=tracker,
    )
    result = use_case.execute(run_name=run_name)
    results[run_name] = result
    m = result.metrics
    print(f"  F1={m.f1:.4f}  Precision={m.precision:.4f}  Recall={m.recall:.4f}  AUC-ROC={m.auc_roc:.4f}")

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {
        "Model": name,
        "F1": r.metrics.f1,
        "Precision": r.metrics.precision,
        "Recall": r.metrics.recall,
        "AUC-ROC": r.metrics.auc_roc,
    }
    for name, r in results.items()
]).set_index("Model").round(4)

print(comparison.to_string())

best_name = comparison["F1"].idxmax()
print(f"\nBest model by F1: {best_name} ({comparison.loc[best_name, 'F1']:.4f})")

## Step 2: SHAP Global Explanation — What Drives Late Deliveries?

SHAP (SHapley Additive exPlanations) decomposes each prediction into feature contributions.
Global importance = mean absolute SHAP value across all test samples.

In [ ]:
# Global feature importance from best model
best_result = results[best_name]
importances = sorted(
    best_result.feature_importances.items(),
    key=lambda x: abs(x[1]),
    reverse=True,
)

print(f"Top features driving late delivery risk ({best_name}):\n")
for i, (feature, importance) in enumerate(importances, 1):
    bar = "\u2588" * int(importance * 100)
    print(f"  {i:2d}. {feature:<35s} {importance:.4f}  {bar}")

## Step 3: MLflow Experiment Tracking

All runs, hyperparameters, metrics, and model artifacts are logged to MLflow.

To view the experiment dashboard:

```
mlflow ui --backend-store-uri sqlite:///mlflow.db
# Open http://localhost:5000
```

The best model is registered in the MLflow Model Registry as `late-delivery-risk`,
ready for loading in the Phase 5 Streamlit dashboard.